# Unconstrained inverse-folding design across models

Runs **unconstrained** (no fixed positions) design on the glyco-benchmark protein set for each inverse-folding model and writes per-model FASTA designs + sequon-retention CSVs.

**Adding a new model**: drop a new adapter at `src/glyco_design/models/<model>.py` that subclasses `DesignModel`, then add a section below. Nothing else should need to change.

**Runtime**: each section assumes a GPU runtime. ProteinMPNN runs fastest (~1–2 min/protein), ESM-IF and TriFlow are slower.

> This notebook is designed to run in Google Colab. Select **Runtime → Change runtime type → GPU** before running.

## Setup (run once)

Clones the SugarFix repo and installs the `glyco_design` package so every model section can import adapters and the pipeline runner.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/LBDillon/SugarFix.git'
REPO_DIR = '/content/SugarFix'
ANALYSIS_DIR = f'{REPO_DIR}/analysis/glycosylation-bias-analysis'
SRC_DIR = f'{ANALYSIS_DIR}/src'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', ANALYSIS_DIR],
    capture_output=True, text=True,
)
if r.returncode != 0:
    print(r.stdout); print(r.stderr)
    raise SystemExit('pip install -e failed')

# Editable installs write a .pth file that Python only reads at startup,
# so add src/ to sys.path so the running kernel sees the package immediately.
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import glyco_design
print('glyco_design ready:', glyco_design.__file__)


In [ ]:
# Shared config for every model section.
import os

ANALYSIS_DIR = '/content/SugarFix/analysis/glycosylation-bias-analysis'
MANIFEST = f'{ANALYSIS_DIR}/data/glyco_benchmark/manifests/benchmark_manifest_simple.csv'
PDB_ROOT = f'{ANALYSIS_DIR}/data/glyco_benchmark/raw'
DESIGN_ROOT = f'{ANALYSIS_DIR}/data/glyco_benchmark/designs'

NUM_SEQS = 32
TEMPERATURE = 0.1
PROTEIN_CLASS = 'glycoprotein'  # set to None to include controls

os.makedirs(DESIGN_ROOT, exist_ok=True)
print('manifest:', MANIFEST)
print('pdb_root:', PDB_ROOT)
print('designs :', DESIGN_ROOT)

## ProteinMPNN

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'git+https://github.com/sokrypton/ColabDesign.git@v1.1.0',
                '--quiet'], check=True)

In [ ]:
from glyco_design.models.proteinmpnn import ProteinMPNNDesignModel
from glyco_design.pipeline import run_unconstrained_experiment

model = ProteinMPNNDesignModel()
model.load()

df_mpnn = run_unconstrained_experiment(
    model=model,
    manifest_path=MANIFEST,
    pdb_root=PDB_ROOT,
    output_dir=f'{DESIGN_ROOT}/proteinmpnn',
    num_seqs=NUM_SEQS,
    temperature=TEMPERATURE,
    protein_class=PROTEIN_CLASS,
)
df_mpnn.head()

## ESM-IF

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'fair-esm[esmfold]', '--quiet'], check=True)

In [ ]:
from glyco_design.models.esm_if import ESMIFDesignModel
from glyco_design.pipeline import run_unconstrained_experiment

model = ESMIFDesignModel(device='cuda')
model.load()

df_esmif = run_unconstrained_experiment(
    model=model,
    manifest_path=MANIFEST,
    pdb_root=PDB_ROOT,
    output_dir=f'{DESIGN_ROOT}/esm_if',
    num_seqs=NUM_SEQS,
    temperature=TEMPERATURE,
    protein_class=PROTEIN_CLASS,
)
df_esmif.head()

## TriFlow

Clones the TriFlow repo and loads the AFDB weights. TriFlow's loader expects weights at a relative path, so the adapter `chdir`s into the repo during `load()`.

In [ ]:
import os, subprocess, sys

TRIFLOW_DIR = '/content/TriFlow'

if not os.path.exists(TRIFLOW_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/jzhoulab/TriFlow.git', TRIFLOW_DIR], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'biopython==1.85', 'pytorch-lightning==2.5.1.post0',
                'deepspeed==0.16.7', 'ml-collections', 'einops', 'dm-tree'], check=True)

assert os.path.exists(f'{TRIFLOW_DIR}/weights/afdb_dataset/afdb_weights.pt'), \
    'TriFlow weights missing — see the TriFlow README for download instructions.'

In [ ]:
from glyco_design.models.triflow import TriFlowDesignModel
from glyco_design.pipeline import run_unconstrained_experiment

model = TriFlowDesignModel(triflow_dir=TRIFLOW_DIR, device='cuda:0')
model.load()

df_triflow = run_unconstrained_experiment(
    model=model,
    manifest_path=MANIFEST,
    pdb_root=PDB_ROOT,
    output_dir=f'{DESIGN_ROOT}/triflow',
    num_seqs=NUM_SEQS,
    temperature=TEMPERATURE,
    protein_class=PROTEIN_CLASS,
)
df_triflow.head()

## Caliby *(TODO)*

Caliby's sampling API is `model.sample(pdb_paths, num_seqs_per_pdb, temperature, pos_constraint_df=...)` — structurally simple to wrap. Adapter at `src/glyco_design/models/caliby.py` is the next step.

## Cross-model comparison

Concatenate per-model retention CSVs for a first-look comparison. Downstream analysis (experiments/04–09) reads from these.

In [ ]:
import pandas as pd

frames = []
for name, df in [('proteinmpnn', df_mpnn), ('esm_if', df_esmif), ('triflow', df_triflow)]:
    if df is not None and len(df):
        frames.append(df)

combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
combined.to_csv(f'{DESIGN_ROOT}/sequon_retention_unconstrained_all_models.csv', index=False)

if len(combined):
    summary = (
        combined.assign(preserved=combined['sequon_status'].isin(['NXS','NXT']))
                .groupby('model')['preserved'].mean()
                .rename('sequon_retention_rate')
                .to_frame()
    )
    summary['n_sites'] = combined.groupby('model').size()
    display(summary)